# Causal Discovery Analysis

Causal search using BOSS/FGES algorithms via PyTetrad.

**Reference:** Kumu R package, `issue_causal_analysis.Rmd` (Stage 5: Causal Search)

## 1. Configuration

Set algorithm parameters and file paths.

In [1]:
# ========================================
# Analysis Parameters
# ========================================
ALGORITHM = "boss"  # Options: "boss" or "fges"
DATA_PATH = "null_variable_dt.csv"
KNOWLEDGE_FILE = "mike_knowledge_box.txt"
OUTPUT_DIR = "boss_results" if ALGORITHM == "boss" else "fges_results"
NON_NULL_OUTPUT_DIR = "boss_domain_results" if ALGORITHM == "boss" else "fges_domain_results"

# Algorithm-specific parameters
PENALTY_DISCOUNT = 2
N_BOOTSTRAP = 100 if ALGORITHM == "boss" else 50
SEED = 32

# Output control
SHOW_BOOTSTRAP_OUTPUT = False  # Set to True to see bootstrap iteration counts

# 1PNEF Threshold percentile (1st percentile = 0.01)
PNEF_PERCENTILE = 0.01

# Nodes of interest for sub-graph analysis (effort variables)
NODES_OF_INTEREST = [
    "silence", "silence2",
    "mis_link", "mis_link2",
    "code_dev", "code_dev2",
    "churn", "churn2",
    "commit", "commit2"
]

## 2. Import Libraries

Load required modules for data processing and causal analysis.

In [2]:
# Increase Java memory allocation for large bootstrap analyses
import os
os.environ['JAVA_TOOL_OPTIONS'] = '-Xmx8g -Xms4g'  # 8GB max heap, 4GB initial
print("✓ Java memory configured: 8GB max, 4GB initial")

✓ Java memory configured: 8GB max, 4GB initial


In [3]:
import pandas as pd
import json
from pykumu_helpers import (
    load_data, detect_variable_types, 
    parse_graph, prepare_non_null_dataset,
    derive_1pnef_threshold, apply_1pnef_threshold
)
from pykumu_algorithms import run_boss_analysis, run_fges_analysis
from pykumu_visualization import (
    prepare_graph_edges, plot_causal_graph, plot_subgraph,
    find_cycles, print_cycle_report
)

Picked up JAVA_TOOL_OPTIONS: -Xmx8g -Xms4g


In [4]:
# Reload modules to pick up code changes
import importlib
import sys
for mod in ['pykumu_algorithms', 'pykumu_helpers', 'pykumu_visualization']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
print("Modules reloaded")

Modules reloaded


## 3. Load Data

Read dataset and validate data quality.

In [5]:
data = load_data(DATA_PATH)
print(f"Dataset: {data.shape[0]} rows × {data.shape[1]} columns")

Dataset: 4870 rows × 138 columns


## 4. Variable Analysis

Identify binary CVE indicators, continuous metrics, and null variables.

In [6]:
var_types = detect_variable_types(data)
print(f"Binary indicators: {len(var_types['cve_indicators'])}")
print(f"Continuous metrics: {len(var_types['metrics'])}")
print(f"Null variables: {len(var_types['null_variables'])}")

Binary indicators: 98
Continuous metrics: 17
Null variables: 23


## 5. Causal Search

Executes causal discovery with bootstrapping and domain knowledge constraints.

In [7]:
# Execute algorithm
if ALGORITHM == "boss":
    results = run_boss_analysis(
        data=data,
        knowledge_file=KNOWLEDGE_FILE,
        output_dir=OUTPUT_DIR,
        penalty_discount=PENALTY_DISCOUNT,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
        suppress_output=not SHOW_BOOTSTRAP_OUTPUT  # Toggle bootstrap output
    )
else:
    results = run_fges_analysis(
        data=data,
        knowledge_file=KNOWLEDGE_FILE,
        output_dir=OUTPUT_DIR,
        penalty_discount=PENALTY_DISCOUNT,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
        suppress_output=not SHOW_BOOTSTRAP_OUTPUT  # Toggle bootstrap output
    )

print(f"\nDiscovered: {results['n_nodes']} nodes, {results['n_edges']} edges")
print(f"Elapsed: {results['elapsed_time']:.1f}s")

BOSS completed: 138 nodes, 94 edges
Elapsed: 54.5s

Discovered: 138 nodes, 94 edges
Elapsed: 54.5s


## 6. Results Summary

Examine discovered edges and their directionality.

In [8]:
edges_df = results['edges']
print(f"Total edges: {len(edges_df)}")
print(f"\nEdge types:")
print(f"  Directed (2→3): {len(edges_df[(edges_df['Endpoint_From']==2) & (edges_df['Endpoint_To']==3)])}")
print(f"  Directed (3→2): {len(edges_df[(edges_df['Endpoint_From']==3) & (edges_df['Endpoint_To']==2)])}")
print(f"  Undirected (3—3): {len(edges_df[(edges_df['Endpoint_From']==3) & (edges_df['Endpoint_To']==3)])}")

print(f"\nFirst 10 edges:")
edges_df.head(10)

Total edges: 94

Edge types:
  Directed (2→3): 36
  Directed (3→2): 58
  Undirected (3—3): 0

First 10 edges:


,From,To,Endpoint_From,Endpoint_To
0,b_160705,start,3,2
1,b_162107,start,3,2
2,b_162180,start,3,2
3,b_173733,nv-b_102939,2,3
4,b_62937,start,2,3
5,b_62940,start,3,2
6,b_63738,start,3,2
7,b_64339,start,3,2
8,b_80891,start,3,2
9,b_81672,start,3,2


## 7. Adjacency Matrix

Matrix representation of graph structure (0=none, 1=circle, 2=arrow, 3=tail).

In [9]:
adjacency = results['adjacency_matrix']
print(f"Adjacency matrix: {adjacency.shape[0]}×{adjacency.shape[1]}")
adjacency.iloc[:5, :5]

Adjacency matrix: 138×138


,b_100433,b_100740,b_100742,b_102939,b_103864
0,0,0,0,0,0
1,0,0,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,0


## 8. Graph String

Text representation of the causal graph.

In [10]:
graph_str = str(results['graph_string'])
print(f"Graph representation ({len(graph_str)} characters):")
print(graph_str[:500] + "...")

Graph representation (13881 characters):
Graph Nodes:
b_100433;b_100740;b_100742;b_102939;b_103864;b_104180;b_113207;b_114109;b_114576;b_114577;b_114619;b_120027;b_120884;b_122110;b_122333;b_130166;b_134353;b_136450;b_140076;b_140160;b_140195;b_140221;b_140224;b_142970;b_143470;b_143505;b_143506;b_143507;b_143508;b_143509;b_143510;b_143511;b_143513;b_143567;b_143568;b_143569;b_143570;b_143571;b_143572;b_148275;b_150204;b_150205;b_150206;b_150207;b_150208;b_150209;b_150285;b_150286;b_150287;b_150288;b_150289;b_150290;b_150291;b_151787;b...


## 9. JSON Representation

JSON format of the causal graph for API integration and visualization tools.

In [11]:
import json

# Get JSON representation from results (already extracted and saved)
graph_json = results['graph_json']

# Parse and pretty-print
json_data = json.loads(str(graph_json))
print(f"JSON structure with {len(json_data.get('nodes', []))} nodes and {len(json_data.get('edges', []))} edges")
print(f"\nFirst 3 nodes:")
print(json.dumps(json_data.get('nodes', [])[:3], indent=2))
print(f"\nFirst 3 edges:")
print(json.dumps(json_data.get('edges', [])[:3], indent=2))
print(f"\nFull JSON ({len(str(graph_json))} characters)")

JSON structure with 138 nodes and 0 edges

First 3 nodes:
[
  {
    "attributes": {},
    "nodeType": "MEASURED",
    "nodeVariableType": "DOMAIN",
    "centerX": -1,
    "centerY": -1,
    "rank": -1,
    "selectionBias": false,
    "name": "b_100433"
  },
  {
    "attributes": {},
    "nodeType": "MEASURED",
    "nodeVariableType": "DOMAIN",
    "centerX": -1,
    "centerY": -1,
    "rank": -1,
    "selectionBias": false,
    "name": "b_100740"
  },
  {
    "attributes": {},
    "nodeType": "MEASURED",
    "nodeVariableType": "DOMAIN",
    "centerX": -1,
    "centerY": -1,
    "rank": -1,
    "selectionBias": false,
    "name": "b_100742"
  }
]

First 3 edges:
[]

Full JSON (7821152 characters)


---

## 10. Null Variable Search Output Files

All null variable search results have been saved to the `{OUTPUT_DIR}` directory:

**Saved files:**
- `adjacency_matrix.csv` - Full 138×138 adjacency matrix (0=none, 1=circle, 2=arrow, 3=tail)
- `edge_list.csv` - Discovered edges with directionality
- `graph_string.txt` - Text representation of the causal graph
- `graph.json` - JSON format for API integration and visualization tools
- `graph.xml` - XML format for Tetrad GUI (load this file in Tetrad to visualize)

---

# Deriving the 1PNEF Threshold

In our causal search above, we introduced null features over multiple bootstrap runs to observe how often our causal search forms random edges (i.e. between our features and null features). We will use this information to derive a threshold, **1PNEF** (1st Percentile NoEdge Frequency), we can use in our final causal search.

## 11. Graph Examination

We parse the Tetrad JSON graph output into tabular format: nodes, edgeset, and edge type probabilities.

The **edgeset** table contains the ensemble edge for each node pair. Because we performed multiple bootstrap runs, the probabilities represent the ensemble of all edges formed on each execution.

The **edge_type_probabilities** table shows the counts of each type of edge formed on each subgraph across all bootstrap runs.

In [12]:
# Parse the JSON output from null variable causal search
null_graph = parse_graph(str(results['graph_json']))

print(f"Nodes: {len(null_graph['nodes'])}")
print(f"\nFirst 5 nodes:")
print(null_graph['nodes'].head())
print(f"\nEdgeset: {len(null_graph['edgeset'])} edges")
print(null_graph['edgeset'].head())
print(f"\nEdge type probabilities: {len(null_graph['edge_type_probabilities'])} entries")
print(null_graph['edge_type_probabilities'].head())

Nodes: 138

First 5 nodes:
  node_name
0  b_100433
1  b_100740
2  b_100742
3  b_102939
4  b_103864

Edgeset: 94 edges
  node1_name node2_name endpoint1 endpoint2   bold  highlighted properties  \
0   code_dev  code_dev2      TAIL     ARROW  False        False        NaN   
1      start   b_160705      TAIL     ARROW  False        False        NaN   
2    silence   silence2      TAIL     ARROW  False        False      dd;pl   
3    silence   mail_dev      TAIL     ARROW  False        False      dd;nl   
4      start    commit2      TAIL     ARROW  False        False      dd;nl   

   probability  
0     1.000000  
1     0.693069  
2     0.970297  
3     1.000000  
4     0.910891  

Edge type probabilities: 298 entries
  node1_name node2_name edge_type properties  probability
0   code_dev  code_dev2        ta        NaN     0.603960
1   code_dev  code_dev2        at        NaN     0.376238
2   code_dev  code_dev2        tt        NaN     0.019802
3      start   b_160705        ta        

## 12. Deriving 1PNEF

Our interest is to derive a threshold for the final causal search, using the information from this bootstrapped null feature causal search. By definition, edges formed between actual variables and random (null) features represent random edges.

We:
1. Subset the edgeset to contain only edges where at least one node is a null variable (nv-*)
2. Derive a `no_edge` probability by subtracting the probability from 1
3. Identify the 1st percentile value of the no_edge probability → the **1PNEF threshold**

This threshold tells us: given entirely random variables, causal links were formed between them up to X% of the time. In our final search, we only keep causal links that formed **more** than X% of the time.

In [13]:
# Derive the 1PNEF threshold from null variable edges
pnef_1, nv_edges = derive_1pnef_threshold(null_graph['edgeset'], percentile=PNEF_PERCENTILE)

print(f"\n1PNEF Threshold: {pnef_1:.4f}")
print(f"\nSample of null variable edges:")
nv_edges.head(10)

Null variable edges: 1
1PNEF threshold (percentile=0.01): 0.4554
  → Random edges formed up to 54.5% of the time
  → Keep only edges with probability > 54.5%

1PNEF Threshold: 0.4554

Sample of null variable edges:


,node1_name,node2_name,endpoint1,endpoint2,bold,highlighted,properties,probability,no_edge
53,b_173733,nv-b_102939,TAIL,ARROW,False,False,pd;pl,0.544554,0.455446


---

# Non-Null Causal Search

With the threshold defined, we now proceed to the final causal search, which **does not include null features**. In this non-null feature causal search, we also specify domain knowledge to prohibit causal links that don't make sense temporally (e.g. features at 1-time-lag cannot cause features in the present).

## 13. Prepare Non-Null Dataset

Remove null variable columns (nv-*) from the dataset, keeping only the original features and binary CVE indicators.

In [14]:
# Prepare dataset without null variables for domain knowledge search
non_null_data = prepare_non_null_dataset(data, save_path="binarized_variable_dt.csv")

Removed 23 null variable columns
Non-null dataset: 4870 rows × 115 columns
Saved to: binarized_variable_dt.csv


## 14. Domain Knowledge Causal Search

Run the causal search on the non-null dataset with domain knowledge constraints. This search uses the same algorithm and bootstrap settings, but on the dataset **without** null features and **with** temporal knowledge constraints.

In [15]:
# Run domain knowledge causal search on non-null dataset
if ALGORITHM == "boss":
    domain_results = run_boss_analysis(
        data=non_null_data,
        knowledge_file=KNOWLEDGE_FILE,
        output_dir=NON_NULL_OUTPUT_DIR,
        penalty_discount=PENALTY_DISCOUNT,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
        suppress_output=not SHOW_BOOTSTRAP_OUTPUT
    )
else:
    domain_results = run_fges_analysis(
        data=non_null_data,
        knowledge_file=KNOWLEDGE_FILE,
        output_dir=NON_NULL_OUTPUT_DIR,
        penalty_discount=PENALTY_DISCOUNT,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
        suppress_output=not SHOW_BOOTSTRAP_OUTPUT
    )

print(f"\nDomain search discovered: {domain_results['n_nodes']} nodes, {domain_results['n_edges']} edges")
print(f"Elapsed: {domain_results['elapsed_time']:.1f}s")

BOSS completed: 115 nodes, 94 edges
Elapsed: 36.4s

Domain search discovered: 115 nodes, 94 edges
Elapsed: 36.4s


## 15. Parse Domain Search Results

Parse the domain knowledge causal search JSON output into nodes, edgeset, and edge type probabilities.

In [16]:
# Parse the domain search JSON output
domain_graph = parse_graph(str(domain_results['graph_json']))

print(f"Domain search nodes: {len(domain_graph['nodes'])}")
print(f"Domain search edges: {len(domain_graph['edgeset'])}")
print(f"\nEdgeset sample:")
domain_graph['edgeset'].head()


Domain search nodes: 115
Domain search edges: 94

Edgeset sample:


,node1_name,node2_name,endpoint1,endpoint2,bold,highlighted,properties,probability
0,code_dev,code_dev2,TAIL,ARROW,False,False,NaN,1.000000
1,start,b_160705,TAIL,ARROW,False,False,NaN,0.693069
2,silence,mail_dev,TAIL,ARROW,False,False,dd;nl,1.000000
3,silence,silence2,TAIL,ARROW,False,False,dd;pl,0.960396
4,churn,commit2,TAIL,ARROW,False,False,NaN,0.811881


---

# Applying 1PNEF Threshold

## 16. Filter Edges

Edges which may have been formed at random are filtered here. We apply the 1PNEF threshold derived from the null variable search to the domain knowledge search results. Only edges whose `no_edge` probability is less than or equal to the 1PNEF threshold are kept.

In [17]:
# Apply 1PNEF threshold to the domain search edges
edges_1pnef = apply_1pnef_threshold(domain_graph['edgeset'], pnef_1)

print(f"\nFiltered edges (1PNEF trimmed):")
edges_1pnef

Edges before threshold: 94
Edges after 1PNEF threshold: 94
Removed: 0 edges (potentially random)

Filtered edges (1PNEF trimmed):


,node1_name,node2_name,endpoint1,endpoint2,bold,highlighted,properties,probability,no_edge
0,code_dev,code_dev2,TAIL,ARROW,False,False,NaN,1.000000,0.000000
1,start,b_160705,TAIL,ARROW,False,False,NaN,0.693069,0.306931
2,silence,mail_dev,TAIL,ARROW,False,False,dd;nl,1.000000,0.000000
3,silence,silence2,TAIL,ARROW,False,False,dd;pl,0.960396,0.039604
4,churn,commit2,TAIL,ARROW,False,False,NaN,0.811881,0.188119
...,...,...,...,...,...,...,...,...,...
89,file,mis_link,TAIL,ARROW,False,False,pd;nl,0.851485,0.148515
90,start,b_64339,TAIL,ARROW,False,False,pd;nl,0.960396,0.039604
91,silence,churn2,TAIL,ARROW,False,False,dd;pl,0.702970,0.297030
92,commit,code_dev,TAIL,ARROW,False,False,pd;nl,0.693069,0.306931


In [18]:
# Save the 1PNEF-filtered edges
import os
os.makedirs(NON_NULL_OUTPUT_DIR, exist_ok=True)
edges_1pnef.to_csv(f"{NON_NULL_OUTPUT_DIR}/edges_1pnef.csv", index=False)
print(f"✓ Saved filtered edges to {NON_NULL_OUTPUT_DIR}/edges_1pnef.csv")

✓ Saved filtered edges to boss_domain_results/edges_1pnef.csv


---

# Results

With the final causal graph trimmed, we can now inspect it to draw conclusions. Causal graphs may form cycles and have undirected edges.

## 17. Full Causal Graph (1PNEF Trimmed)

Interactive visualization of the full causal graph after applying the 1PNEF threshold.

Edge colors:
- **Black**: Directed edges (causal relationship)
- **Red**: Undirected edges (TAIL-TAIL, association without determined direction)

In [19]:
# Prepare edges for visualization
viz_edges = prepare_graph_edges(edges_1pnef)

# Plot the full causal graph
full_graph = plot_causal_graph(
    edges_df=viz_edges,
    graph_nodes=domain_graph['nodes'],
    title="Full Causal Graph (1PNEF Trimmed)",
    output_html=f"{NON_NULL_OUTPUT_DIR}/causal_graph_full.html"
)

# Display in notebook (if pyvis available)
if full_graph is not None:
    full_graph.show(f"{NON_NULL_OUTPUT_DIR}/causal_graph_full.html")

Graph saved to: boss_domain_results/causal_graph_full.html
boss_domain_results/causal_graph_full.html


## 18. Sub-Graph: Effort Variables and Parents

Focus on the key effort variables and their causal relationships. This sub-graph shows only the nodes of interest and edges between them.

In [20]:
# Plot sub-graph for effort variables
sub_graph = plot_subgraph(
    edges_df=viz_edges,
    nodes_of_interest=NODES_OF_INTEREST,
    graph_nodes=domain_graph['nodes'],
    title="Sub-Graph: Effort Variables",
    output_html=f"{NON_NULL_OUTPUT_DIR}/causal_graph_subgraph.html",
    include_parents=False  # Set True to include parent nodes
)

if sub_graph is not None:
    sub_graph.show(f"{NON_NULL_OUTPUT_DIR}/causal_graph_subgraph.html")

Sub-graph: 10 nodes, 31 edges
Graph saved to: boss_domain_results/causal_graph_subgraph.html
boss_domain_results/causal_graph_subgraph.html


## 19. Cycle Detection

Check if the causal graph contains any cycles. Cycles indicate feedback loops in the causal structure.

In [21]:
# Detect cycles in the 1PNEF-trimmed graph
cycles = find_cycles(viz_edges)
print_cycle_report(cycles)

Found 4 cycle(s):
  Cycle 1 (length 3): thread2 → churn2 → churn → thread2
  Cycle 2 (length 4): thread2 → churn2 → churn → file2 → thread2
  Cycle 3 (length 3): thread2 → churn2 → mail_dev2 → thread2
  Cycle 4 (length 3): file2 → churn2 → churn → file2


---

## 20. Summary

**Analysis Pipeline:**
1. **Null Variable Search** — Causal search with null features to observe random edge formation
2. **1PNEF Derivation** — Derived threshold from null variable edges (1st percentile no-edge frequency)
3. **Domain Knowledge Search** — Final search without null features, with temporal constraints
4. **1PNEF Filtering** — Removed edges that may have formed by random chance
5. **Visualization** — Inspected full graph, sub-graphs, and cycles

**Output Files:**
- Null variable search results → `{OUTPUT_DIR}/`
- Domain knowledge search results → `{NON_NULL_OUTPUT_DIR}/`
- 1PNEF-filtered edges → `{NON_NULL_OUTPUT_DIR}/edges_1pnef.csv`
- Interactive graphs → `{NON_NULL_OUTPUT_DIR}/*.html`